[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VectorInstitute/synthetic-data-bootcamp/blob/main/implementations/qa_text_generation/06_dpo_preference_pairs.ipynb)

# Step 6 — DPO Preference Pairs (Optional)

Generate **preference pairs** for Direct Preference Optimization (DPO) that teach
a small model *refusal vs engagement calibration* on the **SEC investor bulletin**
(scope-boundary document).

SFT (notebook 05) copies a single gold answer. DPO instead says: for the same
question, **this** answer is better than **that** one — which is the right tool
when the failure is a tradeoff (too helpful vs too silent, or inventing SEC power).

## Learning objectives
- Build boundary questions from train-split SEC paragraphs only (no test leak)
- Generate four labeled candidates per question (one chosen, three rejected)
- Expand candidates into TRL-style `{prompt, chosen, rejected}` rows
- Optionally run LoRA DPO when CUDA is available

## Prerequisites

1. Run **notebook 01** so `data/paragraphs.jsonl` exists (train/test splits).
2. Teacher API key in `implementations/qa_text_generation/.env` (same as notebooks 02–04).
3. Optional DPO train cell: `uv sync --group text-sft` and `RUN_DPO=1` on an NVIDIA GPU.

This notebook is **SEC-only**. It does not use the CFPB credit-card agreement.

In [2]:
import os
from pathlib import Path

from aieng.syn_data.text import (
    PARAGRAPHS_PATH,
    Paragraph,
    create_judge_client,
    create_teacher_client,
    load_implementation_dotenv,
    load_typed_jsonl,
    save_typed_jsonl,
    use_repo_root,
)
from aieng.syn_data.text.dpo import (
    DEFAULT_DPO_QUESTIONS,
    DPO_ADAPTER_DIR,
    DPO_CANDIDATES_PATH,
    DPO_PAIRS_PATH,
    CalibrationPrompt,
    PreferencePair,
    candidates_to_dpo_pairs,
    filter_pairs_with_judge,
    filter_sec_train_paragraphs,
    generate_boundary_prompts,
    generate_calibration_candidates,
    summarize_rejected_kinds,
    train_lora_dpo,
)
from rich.console import Console
from rich.table import Table


load_implementation_dotenv()
use_repo_root(Path("."))

N_QUESTIONS = int(os.getenv("DPO_N_QUESTIONS", DEFAULT_DPO_QUESTIONS))
VALIDATE_WITH_JUDGE = os.getenv("VALIDATE_WITH_JUDGE", "0") == "1"
RUN_DPO = os.getenv("RUN_DPO", "0") == "1"
BASE_MODEL = os.getenv("DPO_BASE_MODEL", os.getenv("SFT_BASE_MODEL", "Qwen/Qwen2.5-3B-Instruct"))

console = Console(width=100)
console.print(
    f"N_QUESTIONS={N_QUESTIONS}  VALIDATE_WITH_JUDGE={VALIDATE_WITH_JUDGE}  RUN_DPO={RUN_DPO}"
)

N_QUESTIONS=8  VALIDATE_WITH_JUDGE=True  RUN_DPO=True

## Why four candidates?

The SEC bulletin is **investor education** (passphrases, alerts, public Wi-Fi). It does
not tell anyone which stock to buy, and it does not turn optional tips into legal mandates.

For each boundary question we ask the teacher for **one JSON object** with four answers:

| Kind | DPO role | What it does wrong (or right) |
|------|----------|-------------------------------|
| `correctly_scoped` | **chosen** | Answers from the passage; hedges; refuses investment advice |
| `overreaching` | rejected | Gives buy/sell or personal advice the bulletin does not authorize |
| `underreaching` | rejected | Refuses a question the passage *could* answer |
| `authority_misattribution` | rejected | Claims the SEC requires or covers something it does not |

Each question expands to **three** DPO rows (chosen vs each rejected kind).

## 1. Load SEC train paragraphs

Use the **train** split only. Using `test_set.jsonl` here would leak the evaluation set
into preference training.

In [2]:
# Load the previously generated paragraphs.jsonl
all_paragraphs = load_typed_jsonl(PARAGRAPHS_PATH, Paragraph.from_dict)
# Filter the paragraphs to only include the SEC train set
sec_train = filter_sec_train_paragraphs(all_paragraphs)

if not sec_train:
    raise FileNotFoundError(
        f"No SEC train paragraphs in {PARAGRAPHS_PATH}. Run notebook 01 first."
    )

table = Table(title="SEC train paragraphs (scope-boundary)")
table.add_column("#", justify="right")
table.add_column("para_id")
table.add_column("chars", justify="right")
table.add_column("preview")
for i, paragraph in enumerate(sec_train[:8]):
    preview = paragraph.text.replace("\n", " ")[:80] + "…"
    table.add_row(str(i), paragraph.para_id, str(len(paragraph.text)), preview)
console.print(table)
console.print(f"Total SEC train paragraphs: {len(sec_train)}")

                               SEC train paragraphs (scope-boundary)                                
┏━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ # ┃ para_id                      ┃ chars ┃ preview                                               ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 0 │ sec_investor_bulletin::p0000 │   752 │ Updated Investor Bulletin: Protecting Your Online     │
│   │                              │       │ Investment Accounts from Fraud…                       │
│ 1 │ sec_investor_bulletin::p0001 │   795 │ Consider using a “strong” passphrase, instead of a    │
│   │                              │       │ password, if available. Passp…                        │
│ 2 │ sec_investor_bulletin::p0002 │   570 │ If you can’t use a passphrase, pick a “strong”        │
│   │                              │       │ password, keep it secure, and cha…                    │
│ 3 │ sec_investor_bulletin::p0003 │   464 │ What are passkeys? Some investment account websites   │
│   │                              │       │ have started using what is k…                         │
│ 4 │ sec_investor_bulletin::p0007 │   611 │ Account logins Failed account login attempts Password │
│   │                              │       │ changes Personal informati…                           │
│ 5 │ sec_investor_bulletin::p0008 │   512 │ Add biometric safeguards, if available. Your          │
│   │                              │       │ brokerage firm or investment advise…                  │
│ 6 │ sec_investor_bulletin::p0010 │   244 │ Avoid using public computers to access your           │
│   │                              │       │ investment accounts. Avoid accessing…                 │
│ 7 │ sec_investor_bulletin::p0011 │   744 │ Avoid using public computers that require you to      │
│   │                              │       │ enter personal information in o…                      │
└───┴──────────────────────────────┴───────┴───────────────────────────────────────────────────────┘

Total SEC train paragraphs: 11

## 2. Generate boundary questions

The teacher cycles three question types: **in-scope** (should answer with a hedge),
**out-of-scope** (should refuse investment advice), and **gray-boundary** (related to
the bulletin but easy to overclaim SEC authority).

Demo size is small (`N_QUESTIONS`, default 8). Increase it for a real corpus.

In [3]:
teacher = create_teacher_client()
boundary_prompts = generate_boundary_prompts(
    teacher, sec_train, n_questions=N_QUESTIONS
)

q_table = Table(title="Boundary questions")
q_table.add_column("id")
q_table.add_column("kind")
q_table.add_column("question")
for prompt in boundary_prompts:
    q_table.add_row(prompt.id, prompt.question_kind.value, prompt.question[:90])
console.print(q_table)

                                         Boundary questions                                         
┏━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ id           ┃ kind          ┃ question                                                          ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ dpo-4446a9f2 │ in_scope      │ According to the April 23, 2026 Investor Bulletin, what specific  │
│              │               │ types of personal financi                                         │
│ dpo-fe89f7ad │ out_of_scope  │ Based on the bulletin's advice about strong passphrases, which    │
│              │               │ specific password manager s                                       │
│ dpo-59b825d2 │ gray_boundary │ According to the SEC investor-education bulletin, does the SEC    │
│              │               │ legally mandate that all re                                       │
│ dpo-ec554cc6 │ in_scope      │ According to the passage, can you use passkeys on any investment  │
│              │               │ account website or device                                         │
│ dpo-9822b407 │ out_of_scope  │ Based on the list of account alerts like password changes and     │
│              │               │ securities transactions, whi                                      │
│ dpo-6527031f │ gray_boundary │ According to the SEC investor-education bulletin, are brokerage   │
│              │               │ firms legally mandated to                                         │
│ dpo-d9d8ab1b │ in_scope      │ According to the bulletin, what are two specific examples of      │
│              │               │ public computers that investo                                     │
│ dpo-4969d645 │ out_of_scope  │ Since I have to use a public computer today to manage my          │
│              │               │ investments, should I sell my sha                                 │
└──────────────┴───────────────┴───────────────────────────────────────────────────────────────────┘

## 3. Generate four candidates per question

One teacher call returns all four answers so the contrast is internally consistent.

In [4]:
calibration_prompts: list[CalibrationPrompt] = []
for prompt in boundary_prompts:
    try:
        calibration_prompts.append(generate_calibration_candidates(teacher, prompt))
    except (KeyError, ValueError, TypeError, RuntimeError) as exc:
        console.print(f"[yellow]Skipping {prompt.id}: {type(exc).__name__}: {exc}[/yellow]")

example = next(
    (item for item in calibration_prompts if item.candidates),
    None,
)
if example is None:
    raise RuntimeError("Teacher returned no candidates. Check the API key and model.")

console.print(f"[bold]Example question[/bold] ({example.question_kind.value}):")
console.print(example.question)

cand_table = Table(title=f"Candidates for {example.id}")
cand_table.add_column("kind", style="cyan")
cand_table.add_column("role")
cand_table.add_column("answer preview")
cand_table.add_column("rationale")
for candidate in example.candidates:
    role = "chosen" if candidate.kind.value == "correctly_scoped" else "rejected"
    cand_table.add_row(
        candidate.kind.value,
        role,
        candidate.answer.replace("\n", " ")[:180],
        candidate.rationale[:180],
    )
console.print(cand_table)

Example question (in_scope):

According to the April 23, 2026 Investor Bulletin, what specific types of personal financial 
information should investors take steps to safeguard in order to protect their online investment 
accounts from fraud?

                                    Candidates for dpo-4446a9f2                                     
┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ kind                     ┃ role     ┃ answer preview              ┃ rationale                    ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ correctly_scoped         │ chosen   │ According to the bulletin,  │ This answer directly lists   │
│                          │          │ investors should safeguard  │ the specific types of        │
│                          │          │ their personal financial    │ personal financial           │
│                          │          │ information, which includes │ information explicitly       │
│                          │          │ their Social Security       │ mentioned in the passage.    │
│                          │          │ number, financial account   │                              │
│                          │          │ numbers, phone number, e-   │                              │
│ overreaching             │ rejected │ To protect your accounts,   │ This answer overreaches by   │
│                          │          │ you should immediately      │ giving unauthorized personal │
│                          │          │ purchase a premium password │ investment and product       │
│                          │          │ manager and sign up for a   │ recommendations not found in │
│                          │          │ paid identity theft         │ the text.                    │
│                          │          │ monitoring service to       │                              │
│                          │          │ secure your Social Security │                              │
│                          │          │ number                      │                              │
│ underreaching            │ rejected │ The bulletin discusses      │ This answer is underreaching │
│                          │          │ protecting online           │ because it unhelpfully       │
│                          │          │ investment accounts from    │ refuses to provide the       │
│                          │          │ fraud, but I cannot provide │ specific list of information │
│                          │          │ the specific list of        │ that is clearly stated in    │
│                          │          │ personal information you    │ the passage.                 │
│                          │          │ need to safeguard.          │                              │
│ authority_misattribution │ rejected │ The SEC legally mandates    │ This answer misattributes    │
│                          │          │ that all investors must     │ authority by claiming the    │
│                          │          │ immediately register their  │ SEC mandates registration    │
│                          │          │ Social Security numbers and │ and guarantees protection,   │
│                          │          │ financial account numbers   │ which is not supported by    │
│                          │          │ with the SEC's Office of    │ the text.                    │
│                          │          │ Investor Education and As   │                              │
└──────────────────────────┴──────────┴─────────────────────────────┴──────────────────────────────┘

## 4. Expand to DPO rows

Prompts match notebooks 01 and 05 (`DEFAULT_EVAL_SYSTEM` + passage + question) so
preference training uses the same instruction format as SFT and evaluation.

In [5]:
pairs = candidates_to_dpo_pairs(calibration_prompts)
kind_counts = summarize_rejected_kinds(pairs)

count_table = Table(title="Preference pairs by rejected kind")
count_table.add_column("rejected kind")
count_table.add_column("n", justify="right")
for kind, count in kind_counts.items():
    count_table.add_row(kind, str(count))
console.print(count_table)
console.print(f"Total DPO rows: {len(pairs)}")

if pairs:
    sample_pair = pairs[0]
    pair_table = Table(title=f"Example pair {sample_pair.id}")
    pair_table.add_column("field", style="cyan")
    pair_table.add_column("text")
    pair_table.add_row("rejected_kind", sample_pair.rejected_kind.value)
    pair_table.add_row("chosen", sample_pair.chosen[:400])
    pair_table.add_row("rejected", sample_pair.rejected[:400])
    console.print(pair_table)

  Preference pairs by rejected  
              kind              
┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━┓
┃ rejected kind            ┃ n ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━┩
│ overreaching             │ 8 │
│ underreaching            │ 8 │
│ authority_misattribution │ 8 │
└──────────────────────────┴───┘

Total DPO rows: 24

                               Example pair dpo-4446a9f2-overreaching                               
┏━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ field         ┃ text                                                                             ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ rejected_kind │ overreaching                                                                     │
│ chosen        │ According to the bulletin, investors should safeguard their personal financial   │
│               │ information, which includes their Social Security number, financial account      │
│               │ numbers, phone number, e-mail address, and usernames and passwords for online    │
│               │ financial accounts.                                                              │
│ rejected      │ To protect your accounts, you should immediately purchase a premium password     │
│               │ manager and sign up for a paid identity theft monitoring service to secure your  │
│               │ Social Security number and financial account numbers.                            │
└───────────────┴──────────────────────────────────────────────────────────────────────────────────┘

## 5. Optional judge filter and save

Set `VALIDATE_WITH_JUDGE=1` to keep a pair only when the judge prefers chosen over
rejected. Default is off to keep the demo cheap.

In [6]:
if VALIDATE_WITH_JUDGE:
    judge = create_judge_client()
    kept, dropped = filter_pairs_with_judge(judge, calibration_prompts, pairs)
    console.print(f"Judge kept {len(kept)} / {len(pairs)} pairs ({len(dropped)} dropped).")
    pairs = kept
else:
    console.print("Judge validation skipped (VALIDATE_WITH_JUDGE=0).")

save_typed_jsonl(
    DPO_CANDIDATES_PATH,
    calibration_prompts,
    to_dict=CalibrationPrompt.to_dict,
)
save_typed_jsonl(
    DPO_PAIRS_PATH,
    pairs,
    to_dict=PreferencePair.to_dict,
)
console.print(f"Wrote candidates → {DPO_CANDIDATES_PATH}")
console.print(f"Wrote pairs      → {DPO_PAIRS_PATH}")

Judge kept 24 / 24 pairs (0 dropped).

Wrote candidates → 
/home/coder/synthetic-data-bootcamp/implementations/qa_text_generation/data/synthetic/dpo_candidates
.jsonl

Wrote pairs      → 
/home/coder/synthetic-data-bootcamp/implementations/qa_text_generation/data/synthetic/dpo_preference
_pairs.jsonl

## 6. Optional LoRA DPO

Same CUDA / 4-bit LoRA constraints as notebook 05. Leave `RUN_DPO=0` on CPU or
Apple Silicon — you still have a usable preference JSONL.

In [3]:
# Load previously generated DPO pairs
pairs = load_typed_jsonl(DPO_PAIRS_PATH, PreferencePair.from_dict)
print(f"Number of DPO pairs: {len(pairs)}")

Number of DPO pairs: 24


In [4]:
if RUN_DPO:
    adapter_path = train_lora_dpo(
        pairs,
        DPO_ADAPTER_DIR,
        base_model=BASE_MODEL,
        num_train_epochs=1.0,
    )
    console.print(f"[bold green]DPO LoRA adapter saved to {adapter_path}[/bold green]")
else:
    console.print(
        "[bold yellow]DPO training skipped[/bold yellow]\n"
        "Set [green]RUN_DPO=1[/green] on a [green]CUDA[/green] machine to fine-tune.\n"
        f"Preference pairs are ready at {DPO_PAIRS_PATH}"
    )

/home/coder/synthetic-data-bootcamp/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights:   0%|          | 1/434 [00:00<01:09,  6.21it/s]/home/coder/synthetic-data-bootcamp/.venv/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Tokenizing train dataset:   0%|          | 0/24 [00:00<?, ? examples/s][RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+rejected. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of toke

Step,Training Loss


DPO LoRA adapter saved to 
/home/coder/synthetic-data-bootcamp/implementations/qa_text_generation/models/dpo_lora_adapter

## 7. Evaluate DPO on held-out refusal calibration

Load notebook 01's `test_set.jsonl`, keep only `failure_mode == refusal_calibration`, and score the **DPO LoRA adapter** with the same judge as notebooks 01 and 05.

Requires a saved adapter under `models/dpo_lora_adapter` (`RUN_DPO=1` in the previous cell). CUDA is needed for `PeftInferenceClient`.


In [5]:
from aieng.syn_data.text import (
    BASELINE_SCORES_PATH,
    RESULTS_DIR,
    TEST_SET_PATH,
    FailureMode,
    QASample,
    create_judge_client,
    create_small_model_client,
    read_json,
    run_inference,
    save_baseline_results,
    score_predictions,
)
from aieng.syn_data.text.sft import PeftInferenceClient

test_samples = load_typed_jsonl(TEST_SET_PATH, QASample.from_dict)
refusal_samples = [
    sample
    for sample in test_samples
    if sample.failure_mode == FailureMode.REFUSAL_CALIBRATION
]
console.print(
    f"Test set: {len(test_samples)} total, "
    f"{len(refusal_samples)} refusal_calibration"
)
if not refusal_samples:
    raise FileNotFoundError(
        f"No refusal_calibration rows in {TEST_SET_PATH}. Re-run notebook 01."
    )


Test set: 60 total, 20 refusal_calibration

In [6]:

adapter_ready = DPO_ADAPTER_DIR.exists() and any(DPO_ADAPTER_DIR.iterdir())
if adapter_ready:
    eval_client = PeftInferenceClient(DPO_ADAPTER_DIR, BASE_MODEL)
    eval_label = "DPO LoRA"
else:
    raise FileNotFoundError(
        f"RUN_DPO=1 but no adapter at {DPO_ADAPTER_DIR}. Re-run the train cell."
    )

dpo_predictions = run_inference(eval_client, refusal_samples)
judge = create_judge_client()
dpo_scores = score_predictions(judge, refusal_samples, dpo_predictions)

dpo_pred_path = RESULTS_DIR / "dpo_refusal_predictions.jsonl"
dpo_scores_path = RESULTS_DIR / "dpo_refusal_scores.json"
dpo_summary = save_baseline_results(
    dpo_predictions,
    dpo_scores,
    refusal_samples,
    predictions_path=dpo_pred_path,
    scores_path=dpo_scores_path,
)

table = Table(title=f"{eval_label} — refusal_calibration")
table.add_column("Metric", justify="left", style="cyan", no_wrap=True)
table.add_column("Score", justify="right", style="magenta")
for metric, score in dpo_summary["overall"].items():
    table.add_row(metric, f"{score:.3f}")
console.print(table)

if BASELINE_SCORES_PATH.exists():
    baseline_report = read_json(BASELINE_SCORES_PATH)
    baseline_refusal = baseline_report.get("by_failure_mode", {}).get(
        "refusal_calibration"
    )
    if baseline_refusal:
        cmp = Table(title="Baseline vs this run (refusal_calibration)")
        cmp.add_column("Metric", style="cyan")
        cmp.add_column("Baseline", justify="right", style="yellow")
        cmp.add_column(eval_label, justify="right", style="green")
        cmp.add_column("Delta", justify="right", style="magenta")
        for metric, base_val in baseline_refusal.items():
            dpo_val = dpo_summary["overall"].get(metric)
            delta = (dpo_val - base_val) if dpo_val is not None else None
            cmp.add_row(
                metric,
                f"{base_val:.3f}",
                f"{dpo_val:.3f}" if dpo_val is not None else "—",
                f"{delta:+.3f}" if delta is not None else "—",
            )
        console.print(cmp)

console.print(f"Wrote {dpo_pred_path}")
console.print(f"Wrote {dpo_scores_path}")

Loading weights:   0%|          | 1/434 [00:00<01:10,  6.16it/s]/home/coder/synthetic-data-bootcamp/.venv/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 434/434 [00:01<00:00, 229.33it/s]
[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/home/coder/synthetic-data-bootcamp/.venv/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
2026-08-25 01:28:48,230 INFO aieng.syn_data.text.judge: Scoring model answer for sample: test-sec_investor_bulletin::p0004-0 (answer length: 335)
2026-08

 DPO LoRA — refusal_calibration  
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃ Metric                ┃ Score ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━┩
│ correctness           │ 4.700 │
│ coherence             │ 5.000 │
│ instruction_following │ 4.850 │
│ factual_plausibility  │ 4.750 │
│ average               │ 4.825 │
└───────────────────────┴───────┘

       Baseline vs this run (refusal_calibration)       
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┓
┃ Metric                ┃ Baseline ┃ DPO LoRA ┃  Delta ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━┩
│ correctness           │    2.650 │    4.700 │ +2.050 │
│ coherence             │    4.950 │    5.000 │ +0.050 │
│ instruction_following │    3.500 │    4.850 │ +1.350 │
│ factual_plausibility  │    3.300 │    4.750 │ +1.450 │
│ average               │    3.600 │    4.825 │ +1.225 │
└───────────────────────┴──────────┴──────────┴────────┘

Wrote 
/home/coder/synthetic-data-bootcamp/implementations/qa_text_generation/data/results/dpo_refusal_pred
ictions.jsonl

Wrote 
/home/coder/synthetic-data-bootcamp/implementations/qa_text_generation/data/results/dpo_refusal_scor
es.json